# ETL aditivo: dataset extendido

Ejecuta `src/etl_features_ext.py` para generar `data/processed/airprice_features_ext.csv`.

**Estrategia:** el dataset limpio se derivó de `data/listings.csv`, así que se recuperan columnas del original con un merge uno-a-uno por `id` (no se reprocesa todo el CSV). El dataset limpio y `airprice_features.csv` **no se modifican**.

**Qué se agrega:**

- Variables del original: `host_listings_count`, `host_total_listings_count`, `review_scores_*` (6 subpuntuaciones), `availability_30/60/90`, `number_of_reviews_ltm/l30d/ly`, `minimum/maximum_nights_avg_ntm`, `calculated_host_listings_count_*`, `host_since`, `first_review`, `last_review`, `amenities`.
- Derivadas: antigüedad del host, días desde reviews, ratios de disponibilidad y reviews, ratios de portafolio del host, rangos de noches, ratios de capacidad.
- Amenidades: ~50 flags binarias (`amenity_wifi`, `amenity_elevator`, `amenity_pool`, etc.).
- Geo adicional: distancia mínima a POIs, a estaciones, a landmarks y a aeropuertos; número de POIs a 2 km; densidad de anuncios a 1 km, distancia al vecino más cercano, densidad por vecindario/cluster; K-Means con k=5, 8 y 10 ajustado solo con train.

**Fuga de datos:** se excluyen `price`, `price_quote_*`, `estimated_revenue_l365d` y `estimated_occupancy_l365d`.

In [ ]:
import sys
sys.path.append('../..')

import pandas as pd

from src.etl_features_ext import (
    AUDIT_PATH,
    METADATA_PATH,
    SALIDA_PATH,
    construir_dataset_ext,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

In [ ]:
df = construir_dataset_ext()
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
print('Guardado en:', SALIDA_PATH)
print('Auditoría:', AUDIT_PATH)
print('Metadatos:', METADATA_PATH)

In [ ]:
auditoria = pd.read_csv(AUDIT_PATH)
display(auditoria.T)

print('Tasa de emparejamiento por id:', float(auditoria['tasa_emparejamiento_id'].iloc[0]))
print('Columnas nuevas:', int(auditoria['columnas_nuevas'].iloc[0]))

In [ ]:
columnas_geo = [
    c for c in df.columns
    if c.startswith('distancia_') or c.startswith('cluster_zona') or c in (
        'vecinos_1km', 'pois_2km', 'densidad_vecindario', 'ratio_densidad_1km'
    )
]
print('Columnas geo extendidas:', columnas_geo)
print('Flags de amenidades:', [c for c in df.columns if c.startswith('amenity_')][:10], '...')

df[['id', 'price_range'] + columnas_geo].head()

In [ ]:
nulos = (
    df.isna().mean()
    .sort_values(ascending=False)
    .head(20)
    .rename('proporcion_nulos')
    .to_frame()
)
display(nulos)

print('Duplicados de id:', int(df['id'].duplicated().sum()))
print('price_clean en el dataset extendido:', 'price_clean' in df.columns)
print('price en el dataset extendido:', 'price' in df.columns)
print('estimated_revenue_l365d en el dataset extendido:', 'estimated_revenue_l365d' in df.columns)

## Notas de implementación

- El merge conserva el orden de `airprice_features.csv`, por lo que el split estratificado es idéntico al de los escenarios base y geo.
- Los K-Means se ajustan **solo con las coordenadas de train**; luego se asignan todos los registros. Así se evita usar información del test para construir Features.
- La densidad (`vecinos_1km`, `distancia_vecino_km`) se calcula con `BallTree` y distancia haversine. Es una propiedad del territorio, no del target; no hay fuga.
- Algunas variables del plan original no existen en esta versión del scrape (90 columnas): `guests_included`, `extra_people`, `security_deposit`, `cleaning_fee` no están disponibles. No se inventan.
- `estimated_occupancy_l365d` se excluye por ser una estimación derivada del historial, no una característica observable al publicar el anuncio.